# Customer Churn Prediction

*Exploratory Data Analysis · Predictive Modeling · Model Evaluation*

---

This study examines customer attrition using the historical Cell2Cell US telecommunications dataset. It explores customer attributes, compares supervised classification approaches, and evaluates how effectively predicted risk distinguishes subscribers who churn from those who stay.

**Research objective:** identify likely churners while understanding the trade-off between missed customers and unnecessary retention outreach.

**Data:** 51,047 historical customer records. The source dataset must be present locally to reproduce the figures. Measured holdout results are documented in `../docs/BENCHMARK.md`.

### Study roadmap

1. Research question and data
2. Exploratory data analysis and quality checks
3. Feature relationships and modeling
4. Model evaluation and visual diagnostics
5. Interpretation, limitations, and conclusions

---

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / "churn").is_dir(): ROOT = ROOT.parent
assert (ROOT / "churn").is_dir(), "Open Jupyter from project root or notebooks/"
sys.path.insert(0, str(ROOT))
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from churn.train import load_real_data, run, SEED
DATA = ROOT / "data" / "raw" / "cell2celltrain.csv"
X, y, sha = load_real_data(DATA)
print("Rows:",len(X),"Features:",X.shape[1],"SHA-256:",sha)
display(X.head())

## Extended analysis — questions and hypotheses

The practical question is **which subscribers might churn**, not merely whether accuracy is high. Before fitting models, investigate: (1) overall churn rate; (2) missingness and suspicious identifier/target columns; (3) how numerical and categorical attributes relate to churn; (4) whether the model performs better than a prevalence-only baseline; (5) how many false alerts an outreach team would receive. Associations are descriptive, not causal. All figures below are calculated from the actual downloaded CSV.

In [ ]:
# Customer-level data quality overview, including cardinality and null values
quality = pd.DataFrame({
    "dtype": X.dtypes.astype(str),
    "missing_n": X.isna().sum(),
    "missing_pct": (X.isna().mean()*100).round(2),
    "unique_values": X.nunique(dropna=True)
}).sort_values("missing_pct", ascending=False)
display(quality.head(30))
print("Duplicate feature rows:",int(X.duplicated().sum()),"of",len(X))
print("Outcome distribution:",y.value_counts().sort_index().to_dict())

## 1. Explore the target distribution and data quality
The true churn prevalence is the relevant baseline for average precision.

In [ ]:
print("Churn counts and percentages:")
display(pd.DataFrame({"count":y.value_counts().sort_index(),"percent":(100*y.value_counts(normalize=True).sort_index()).round(2)}))
display(X.isna().mean().sort_values(ascending=False).head(15).rename("missing_fraction").to_frame())
fig, ax = plt.subplots(figsize=(5,3))
y.map({0:"Retained",1:"Churned"}).value_counts().plot.bar(ax=ax)
ax.set_title("Real Cell2Cell customer outcome counts")
plt.tight_layout(); plt.show()

## 2. Explore numeric customer features
Plots are descriptive only. Never infer causality from these observational associations.

In [ ]:
num_cols = X.select_dtypes(include="number").columns.tolist()
display(X[num_cols].describe().T.head(20))
if num_cols:
    ax = X[num_cols[:min(4,len(num_cols))]].hist(figsize=(11,7),bins=25)
    plt.suptitle("Numeric feature distributions"); plt.tight_layout(); plt.show()

## Numeric feature relationships

Compare retained and churned subscribers using *real* observations. Median differences do not imply that manipulating a feature would change a subscriber's decision.

In [ ]:
numeric = X.select_dtypes(include="number")
ranked = numeric.columns[numeric.notna().sum().gt(len(X)*0.75)].tolist()
if not ranked: raise ValueError("No well-observed numeric columns to plot")
fig, axes = plt.subplots(min(4,len(ranked)),1,figsize=(9,3*min(4,len(ranked))))
axes = np.atleast_1d(axes)
for ax, name in zip(axes,ranked[:4]):
    values = pd.DataFrame({"feature":numeric[name],"churn":y})
    values.boxplot(column="feature",by="churn",ax=ax,showfliers=False)
    ax.set_title(name+" by actual churn class");ax.set_xlabel("Churn (0=no,1=yes)")
plt.suptitle("");plt.tight_layout();plt.show()

In [ ]:
# Correlations describe association, not feature importance or causality.
corr = numeric.select_dtypes(include="number").corr(numeric_only=True)
selected = corr.columns[:min(14,len(corr))]
fig,ax=plt.subplots(figsize=(10,8))
matrix=corr.loc[selected,selected].to_numpy()
im=ax.imshow(matrix,interpolation="nearest",vmin=-1,vmax=1,cmap="coolwarm")
ax.set_xticks(range(len(selected)),selected,rotation=90)
ax.set_yticks(range(len(selected)),selected)
fig.colorbar(im,ax=ax,label="Pearson correlation")
ax.set_title("Exploratory numeric feature correlations")
plt.tight_layout();plt.show()

## Categorical feature relationships

Category prevalence can be misleading when some groups are tiny. Show the most frequent categories and explicitly display their sample sizes.

In [ ]:
categories=X.select_dtypes(exclude="number").columns.tolist()
if categories:
    name=categories[0]
    table=pd.DataFrame({"category":X[name].astype("string").fillna("<missing>"),"churn":y})
    common=table["category"].value_counts().head(12).index
    group=table[table["category"].isin(common)].groupby("category")["churn"].agg(["mean","size"]).sort_values("mean")
    display(group.rename(columns={"mean":"churn_rate","size":"customer_count"}))
    fig,ax=plt.subplots(figsize=(9,4))
    group["mean"].plot.barh(ax=ax)
    ax.set_xlabel("Observed churn rate");ax.set_title(f"{name}: group comparison")
    plt.tight_layout();plt.show()
else:
    print("No categorical predictors in this dataset")

## 3. Train and compare models
The production-oriented pipeline handles stratified 60/20/20 splitting, train-only imputation and encoding, validation average-precision selection, validation-only threshold optimization, and one frozen test evaluation. Training here writes only local files in `artifacts/`.

In [ ]:
from churn.train import run
results = run(DATA, ROOT / "artifacts")
comparison = pd.DataFrame(results["validation"]).T
display(comparison[["average_precision","roc_auc","precision","recall","f1"]].sort_values("average_precision",ascending=False))
print("Winner:",results["selected_model"],"threshold:",round(results["threshold"],3))

## Why average precision rather than accuracy?

In an imbalanced classification problem, predicting the majority class for every customer can look accurate but identifies no churners. The validation selection metric here is average precision (AP), which summarizes the ranking quality of precision and recall. A prevalence predictor's AP is near churn prevalence.

In [ ]:
validation = pd.DataFrame(results["validation"]).T.sort_values("average_precision",ascending=False)
fig,axes=plt.subplots(1,2,figsize=(11,4))
validation["average_precision"].plot.bar(ax=axes[0],title="Validation average precision")
validation["roc_auc"].plot.bar(ax=axes[1],title="Validation ROC-AUC")
for ax in axes: ax.set_ylim(0,1);ax.set_xlabel("");ax.tick_params(axis="x",rotation=25)
plt.tight_layout();plt.show()
display(validation[["average_precision","roc_auc","brier_score","precision","recall","f1"]])

## 4. Held-out test evaluation and error analysis
These test results must not drive any further selection.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, PrecisionRecallDisplay, RocCurveDisplay
from sklearn.metrics import precision_recall_curve, roc_curve
out = pd.read_csv(ROOT / "artifacts" / "test_predictions.csv")
display(pd.DataFrame([results["test"]]).drop(columns="confusion_matrix").T.rename(columns={0:"test"}))
from sklearn.metrics import confusion_matrix
fig,axes=plt.subplots(1,2,figsize=(10,4))
ConfusionMatrixDisplay.from_predictions(out["actual_churn"],out["predicted_churn"],ax=axes[0],colorbar=False)
PrecisionRecallDisplay.from_predictions(out["actual_churn"],out["churn_probability"],ax=axes[1])
plt.tight_layout();plt.show()

## Threshold and resource trade-off

The fitted classifier produces churn probabilities; decisions require a threshold. Changing the threshold changes precision, recall, and the number of subscribers flagged. The published selected threshold was determined on validation data. The following *test* sweep is **diagnostic only**—do not pick a new threshold using test data.

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score
diagnostic_thresholds=np.linspace(.1,.9,17)
rows=[]
for t in diagnostic_thresholds:
    p=(out["churn_probability"].to_numpy()>=t).astype(int)
    rows.append({"threshold":t,"precision":precision_score(out["actual_churn"],p,zero_division=0),
                 "recall":recall_score(out["actual_churn"],p,zero_division=0),
                 "f1":f1_score(out["actual_churn"],p,zero_division=0),
                 "flagged":int(p.sum())})
sensitivity=pd.DataFrame(rows)
display(sensitivity)
ax=sensitivity.plot(x="threshold",y=["precision","recall","f1"],figsize=(9,4))
ax.axvline(results["threshold"],ls="--",color="black",label="Frozen validation threshold")
ax.set_ylabel("Score");ax.set_title("Test-set sensitivity analysis (NOT tuning)")
plt.tight_layout();plt.show()

## Calibration and probability quality

A model can rank customers well yet produce poorly calibrated probabilities. We report Brier score and visualize the difference between average predicted risk and observed churn rate, but this is only a diagnostic on the held-out split.

In [ ]:
from sklearn.calibration import calibration_curve
fraction,mean_pred=calibration_curve(out["actual_churn"],out["churn_probability"],n_bins=10,strategy="quantile")
fig,ax=plt.subplots(figsize=(5,5))
ax.plot([0,1],[0,1],"k--",label="Perfect calibration")
ax.plot(mean_pred,fraction,"o-",label=results["selected_model"])
ax.set(xlabel="Average predicted churn probability",ylabel="Observed churn fraction",
       title="Held-out calibration diagnostic",xlim=(0,1),ylim=(0,1))
ax.legend();plt.tight_layout();plt.show()

## Error analysis and business context

False positives waste outreach capacity; false negatives miss churners. A model with high recall is not automatically profitable. No campaign-level intervention effects or retention costs were supplied, so we should not estimate ROI.

In [ ]:
report=out.copy()
report["error_type"]=np.select([
    (report.actual_churn==1)&(report.predicted_churn==0),
    (report.actual_churn==0)&(report.predicted_churn==1)],
    ["missed churner","false retention alert"],default="correct")
display(report["error_type"].value_counts().rename("customers").to_frame())
print("Dataset SHA256:",results["dataset_sha256"])
print("Best model:",results["selected_model"],"| Test AP:",round(results["test"]["average_precision"],4))
print("Conclusion: historical benchmark, not evidence of retention lift or production reliability.")

## Conclusion and next steps

The trained model is evaluated on historical, held-out Cell2Cell customer records. High recall comes with substantial false-positive outreach: interpret precision, average precision, calibration, and capacity constraints together. A random stratified split does not establish performance on future customers.

---

**Further development:** A separate software-engineering implementation of this project is available in the repository, with reusable training code, automated checks, and saved model artifacts.